# Home Loan Default Risk & Customer Eligibility — 

**Domain: Banking / Credit Risk**

This notebook is a memory-conscious implementation of the supplied Home Loan Default project.

### Models used
1. **Logistic Regression** — interpretable baseline.
2. **XGBoost** — nonlinear gradient-boosted tree model.

###  design principles
- Process historical CSV files one at a time.
- Load only required columns.
- Create compact applicant-level aggregates.
- Avoid direct expansion of `bureau_balance.csv`.
- Use a curated numeric feature set rather than every engineered column.
- Avoid large one-hot encoded matrices.
- Use conservative XGBoost settings (`hist`, shallow trees, small `max_bin`, limited threads).
- Report ROC-AUC, PR-AUC, accuracy, precision, recall and F1.
- Perform threshold analysis and customer risk segmentation.

The original project asks for complete data analysis, a predictive model for customer eligibility, model comparison and a challenges report. This notebook retains those requirements while replacing HistGradientBoosting with a memory-constrained XGBoost model.


## 1. Dataset

The project uses the Home Credit-style relational dataset:

- `application_train.csv` — main application table and target.
- `bureau.csv` — previous credits reported by other financial institutions.
- `bureau_balance.csv` — monthly bureau credit history.
- `POS_CASH_balance.csv` — previous POS/cash-loan monthly snapshots.
- `credit_card_balance.csv` — previous credit-card monthly snapshots.
- `previous_application.csv` — previous Home Credit applications.
- `installments_payments.csv` — previous installment repayment history.

These definitions and the target meaning come directly from the supplied project description. fileciteturn0file0L10-L24 fileciteturn0file0L25-L51

In [ ]:
# 2. Imports and configuration
import gc
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score,
    precision_score, recall_score, f1_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve
)

# XGBoost
try:
    from xgboost import XGBClassifier
except ImportError as e:
    raise ImportError(
        "XGBoost is not installed. Run this once in a notebook cell:\n"
        "%pip install xgboost\n"
        "Then restart the kernel and run the notebook again."
    ) from e

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

RANDOM_STATE = 42
TEST_SIZE = 0.20

# Keep CPU parallelism deliberately low for an 8 GB laptop.
XGB_N_JOBS = 2

# Put the ZIP or extracted CSV files beside this notebook.
ZIP_PATH = Path("PRCP-1006-HomeLoanDef.zip")
DATA_DIR = Path("data")

if ZIP_PATH.exists():
    EXTRACT_DIR = Path("home_loan_data")
    EXTRACT_DIR.mkdir(exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(EXTRACT_DIR)
    DATA_DIR = EXTRACT_DIR

print("Data directory:", DATA_DIR.resolve())
print("XGBoost version:", __import__("xgboost").__version__)
print("CPU threads for XGBoost:", XGB_N_JOBS)


In [ ]:
# 3. Locate the seven CSV files
EXPECTED = [
    "application_train.csv",
    "bureau.csv",
    "bureau_balance.csv",
    "POS_CASH_balance.csv",
    "credit_card_balance.csv",
    "previous_application.csv",
    "installments_payments.csv",
]

def locate_file(filename):
    matches = list(Path(".").rglob(filename))
    if not matches:
        raise FileNotFoundError(
            f"{filename} not found. Put the CSV files in ./data/ "
            "or put PRCP-1006-HomeLoanDef.zip beside the notebook."
        )
    return matches[0]

paths = {name: locate_file(name) for name in EXPECTED}
for k, v in paths.items():
    print(f"{k:28s} -> {v}")

In [ ]:
# 4. Load only the main application table
application = pd.read_csv(paths["application_train.csv"])

print("Application shape:", application.shape)
print("Target distribution:")
display(application["TARGET"].value_counts(normalize=False).rename("count").to_frame())
display((application["TARGET"].value_counts(normalize=True) * 100).round(2).rename("percentage").to_frame())

gc.collect()

## 5. Data quality and target analysis

In [ ]:
# Missing values and duplicate applicants
missing = (
    application.isna().mean()
    .mul(100)
    .sort_values(ascending=False)
    .head(20)
    .rename("missing_percent")
    .to_frame()
)
display(missing)

print("Duplicate SK_ID_CURR:", application["SK_ID_CURR"].duplicated().sum())
print("Default rate:", round(application["TARGET"].mean() * 100, 2), "%")

plt.figure(figsize=(5, 4))
sns.countplot(data=application, x="TARGET")
plt.title("Loan Default Distribution")
plt.xlabel("TARGET (1 = Default, 0 = Non-default)")
plt.show()

## 6. Lightweight exploratory data analysis

Only a small set of high-value application variables is used for the main EDA. This avoids creating dozens of unnecessary plots and keeps the notebook responsive on an 8 GB machine.

In [ ]:
eda_cols = [
    "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE",
    "DAYS_BIRTH", "DAYS_EMPLOYED", "EXT_SOURCE_1", "EXT_SOURCE_2",
    "EXT_SOURCE_3", "CNT_CHILDREN", "CNT_FAM_MEMBERS",
    "REGION_RATING_CLIENT", "OWN_CAR_AGE"
]
eda_cols = [c for c in eda_cols if c in application.columns]

eda = application[["TARGET"] + eda_cols].copy()
if "DAYS_BIRTH" in eda:
    eda["AGE_YEARS"] = eda["DAYS_BIRTH"].abs() / 365.25
if "DAYS_EMPLOYED" in eda:
    eda["EMPLOYMENT_YEARS"] = np.where(
        eda["DAYS_EMPLOYED"] < 0,
        eda["DAYS_EMPLOYED"].abs() / 365.25,
        np.nan
    )

display(eda.describe().T)

for c in [x for x in ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AGE_YEARS",
                      "EXT_SOURCE_2", "EXT_SOURCE_3"] if x in eda.columns]:
    plt.figure(figsize=(6, 4))
    sns.boxplot(data=eda, x="TARGET", y=c)
    plt.title(f"{c} vs Default")
    plt.tight_layout()
    plt.show()

In [ ]:
# Default rate by a few important categorical variables
cat_cols = [
    "NAME_CONTRACT_TYPE", "CODE_GENDER", "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS", "NAME_INCOME_TYPE", "NAME_HOUSING_TYPE"
]
for c in [x for x in cat_cols if x in application.columns]:
    tmp = application.groupby(c)["TARGET"].agg(["mean", "count"])
    tmp = tmp[tmp["count"] >= max(100, int(len(application) * 0.002))]
    print(f"\nDefault rate by {c}")
    display(tmp.sort_values("mean", ascending=False).head(10).rename(columns={"mean": "default_rate"}))

## 7. Memory-efficient historical feature engineering

The original notebook aggregates all numeric columns with several statistics (`mean`, `min`, `max`, `std`, `sum`) and keeps several large intermediate tables in memory. That is the main risk for an 8 GB laptop.

Here we use **small, business-relevant aggregates** and process each historical file independently. The raw file is deleted after its features are created.

In [ ]:
# Helper: compact numeric aggregation
def compact_aggregate(df, key="SK_ID_CURR", prefix=""):
    numeric_cols = [c for c in df.select_dtypes(include=np.number).columns if c != key]
    if not numeric_cols:
        return df[[key]].drop_duplicates()

    agg = {}
    for c in numeric_cols:
        agg[c] = ["mean", "max", "min"]
    out = df.groupby(key).agg(agg)
    out.columns = [f"{prefix}{c}_{stat}" for c, stat in out.columns]
    return out.reset_index()

def add_features(base, feature_df):
    return base.merge(feature_df, on="SK_ID_CURR", how="left")

In [ ]:
# 7A. Bureau features — selected columns only
bureau_cols = [
    "SK_ID_CURR", "DAYS_CREDIT", "CREDIT_DAY_OVERDUE",
    "AMT_CREDIT_SUM", "AMT_CREDIT_SUM_DEBT",
    "AMT_CREDIT_SUM_OVERDUE", "CNT_CREDIT_PROLONG",
    "AMT_ANNUITY"
]
bureau_cols = [c for c in bureau_cols if c in pd.read_csv(paths["bureau.csv"], nrows=0).columns]

bureau = pd.read_csv(paths["bureau.csv"], usecols=bureau_cols)
bureau_features = compact_aggregate(bureau, prefix="bureau_")
bureau_features["bureau_credit_count"] = bureau.groupby("SK_ID_CURR").size().values
print("Bureau features:", bureau_features.shape)

del bureau
gc.collect()

In [ ]:
# 7B. Previous application features
prev_cols = [
    "SK_ID_CURR", "AMT_ANNUITY", "AMT_APPLICATION", "AMT_CREDIT",
    "AMT_DOWN_PAYMENT", "AMT_GOODS_PRICE", "DAYS_DECISION",
    "CNT_PAYMENT"
]
available = pd.read_csv(paths["previous_application.csv"], nrows=0).columns
prev_cols = [c for c in prev_cols if c in available]

previous = pd.read_csv(paths["previous_application.csv"], usecols=prev_cols)
previous_features = compact_aggregate(previous, prefix="prev_")
previous_features["previous_application_count"] = previous.groupby("SK_ID_CURR").size().values

del previous
gc.collect()
print("Previous-application features:", previous_features.shape)

In [ ]:
# 7C. POS/CASH features
pos_cols = ["SK_ID_CURR", "MONTHS_BALANCE", "CNT_INSTALMENT",
            "CNT_INSTALMENT_FUTURE", "SK_DPD", "SK_DPD_DEF"]
available = pd.read_csv(paths["POS_CASH_balance.csv"], nrows=0).columns
pos_cols = [c for c in pos_cols if c in available]

pos = pd.read_csv(paths["POS_CASH_balance.csv"], usecols=pos_cols)
pos_features = compact_aggregate(pos, prefix="pos_")
pos_features["pos_record_count"] = pos.groupby("SK_ID_CURR").size().values

del pos
gc.collect()
print("POS/CASH features:", pos_features.shape)

In [ ]:
# 7D. Credit-card features
cc_cols = [
    "SK_ID_CURR", "MONTHS_BALANCE", "AMT_BALANCE",
    "AMT_CREDIT_LIMIT_ACTUAL", "AMT_DRAWINGS_ATM",
    "AMT_DRAWINGS_CURRENT", "AMT_PAYMENT_TOTAL_CURRENT",
    "AMT_RECEIVABLE_PRINCIPAL", "SK_DPD", "SK_DPD_DEF"
]
available = pd.read_csv(paths["credit_card_balance.csv"], nrows=0).columns
cc_cols = [c for c in cc_cols if c in available]

credit_card = pd.read_csv(paths["credit_card_balance.csv"], usecols=cc_cols)
cc_features = compact_aggregate(credit_card, prefix="cc_")
cc_features["cc_record_count"] = credit_card.groupby("SK_ID_CURR").size().values

del credit_card
gc.collect()
print("Credit-card features:", cc_features.shape)

In [ ]:
# 7E. Installment repayment behavior
inst_cols = [
    "SK_ID_CURR", "DAYS_INSTALMENT", "DAYS_ENTRY_PAYMENT",
    "AMT_INSTALMENT", "AMT_PAYMENT"
]
available = pd.read_csv(paths["installments_payments.csv"], nrows=0).columns
inst_cols = [c for c in inst_cols if c in available]

installments = pd.read_csv(paths["installments_payments.csv"], usecols=inst_cols)

if {"DAYS_INSTALMENT", "DAYS_ENTRY_PAYMENT"}.issubset(installments.columns):
    installments["payment_delay"] = (
        installments["DAYS_ENTRY_PAYMENT"] - installments["DAYS_INSTALMENT"]
    )

if {"AMT_INSTALMENT", "AMT_PAYMENT"}.issubset(installments.columns):
    installments["payment_shortfall"] = (
        installments["AMT_INSTALMENT"] - installments["AMT_PAYMENT"]
    )

installment_features = compact_aggregate(installments, prefix="inst_")
installment_features["installment_record_count"] = installments.groupby("SK_ID_CURR").size().values

del installments
gc.collect()
print("Installment features:", installment_features.shape)

### 7F. Bureau balance

`bureau_balance.csv` is intentionally not loaded into the model in this 8 GB version because it is a very large monthly-history table. The project description identifies it as monthly history for previous bureau credits. Its information is partly represented through the compact bureau-level aggregates above, while excluding this extra expansion substantially reduces peak memory usage. fileciteturn0file0L19-L24

## 8. Build the applicant-level modeling table

In [ ]:
model_df = application.copy()

for ft in [
    bureau_features, previous_features, pos_features,
    cc_features, installment_features
]:
    model_df = add_features(model_df, ft)

# Release feature tables after merging
del bureau_features, previous_features, pos_features, cc_features, installment_features
gc.collect()

# Application-level ratios
def safe_ratio(a, b):
    return np.where(b.notna() & (b != 0), a / b, np.nan)

if {"AMT_CREDIT", "AMT_INCOME_TOTAL"}.issubset(model_df.columns):
    model_df["CREDIT_TO_INCOME"] = safe_ratio(
        model_df["AMT_CREDIT"], model_df["AMT_INCOME_TOTAL"]
    )

if {"AMT_ANNUITY", "AMT_INCOME_TOTAL"}.issubset(model_df.columns):
    model_df["ANNUITY_TO_INCOME"] = safe_ratio(
        model_df["AMT_ANNUITY"], model_df["AMT_INCOME_TOTAL"]
    )

if "DAYS_BIRTH" in model_df.columns:
    model_df["AGE_YEARS"] = model_df["DAYS_BIRTH"].abs() / 365.25

if "DAYS_EMPLOYED" in model_df.columns:
    model_df["EMPLOYMENT_YEARS"] = np.where(
        model_df["DAYS_EMPLOYED"] < 0,
        model_df["DAYS_EMPLOYED"].abs() / 365.25,
        np.nan
    )

print("Final modeling table:", model_df.shape)
print("Duplicate applicant IDs:", model_df["SK_ID_CURR"].duplicated().sum())

## 9. Train/test split and memory-safe feature selection

For an 8 GB laptop, the model uses numeric variables only. This avoids a huge sparse/dense one-hot matrix from the many categorical fields. The categorical variables remain part of the EDA/business analysis.

In [ ]:
# Memory-safe curated modeling dataset
# We intentionally do NOT use every numeric column.
# A smaller feature set keeps both Logistic Regression and XGBoost practical on 8 GB RAM.

preferred_features = [
    # Application / affordability
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",

    # Household / profile
    "CNT_CHILDREN",
    "CNT_FAM_MEMBERS",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",

    # External risk scores
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",

    # Regional risk indicators
    "REGION_RATING_CLIENT",
    "REGION_RATING_CLIENT_W_CITY",

    # Application-level ratios
    "CREDIT_TO_INCOME",
    "ANNUITY_TO_INCOME",
    "AGE_YEARS",
    "EMPLOYMENT_YEARS",

    # Bureau
    "bureau_credit_count",
    "bureau_AMT_CREDIT_SUM_mean",
    "bureau_AMT_CREDIT_SUM_DEBT_mean",
    "bureau_AMT_CREDIT_SUM_OVERDUE_max",

    # Previous applications
    "previous_application_count",
    "prev_AMT_CREDIT_mean",
    "prev_AMT_APPLICATION_mean",

    # POS / cash
    "pos_SK_DPD_mean",
    "pos_SK_DPD_DEF_mean",
    "pos_record_count",

    # Credit card
    "cc_AMT_BALANCE_mean",
    "cc_AMT_CREDIT_LIMIT_ACTUAL_mean",
    "cc_SK_DPD_mean",

    # Installments
    "inst_payment_delay_mean",
    "inst_payment_shortfall_mean",
    "installment_record_count"
]

selected_features = [c for c in preferred_features if c in model_df.columns]
missing_features = [c for c in preferred_features if c not in model_df.columns]

X = model_df[selected_features].copy()
y = model_df["TARGET"].astype("int8")

# float32 cuts numeric memory roughly in half versus float64.
X = X.astype("float32")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Selected features:", len(selected_features))
print("Missing requested features:", missing_features)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train default rate:", round(y_train.mean(), 4))
print("Test default rate :", round(y_test.mean(), 4))
print("Train memory (MB):", round(X_train.memory_usage(deep=True).sum() / 1024**2, 2))


## 10. Preprocessing

In [ ]:
# Preprocessing
# Logistic Regression needs imputation + scaling.
# XGBoost can natively handle NaN values, so we deliberately avoid
# an extra imputation copy for the tree model.

linear_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

tree_preprocessor = "passthrough"

print("Logistic Regression: median imputation + standardization")
print("XGBoost: native missing-value handling (no imputer)")


## 11. Model comparison — Logistic Regression vs XGBoost

Two complementary models are used:

1. **Logistic Regression** — interpretable linear baseline and benchmark for credit-risk classification.
2. **XGBoost** — nonlinear gradient-boosted tree model that can capture interactions and nonlinear relationships.

### Why XGBoost instead of HistGradientBoosting?
The original optimized notebook used HistGradientBoosting to keep memory usage low. This version replaces it with XGBoost while keeping the same 8 GB design philosophy.

The XGBoost configuration is intentionally conservative:
- `tree_method="hist"`
- shallow trees (`max_depth=3`)
- limited estimators
- `max_bin=64`
- row/column subsampling
- regularization
- only 2 CPU threads
- no large hyperparameter search

This is **not** a full-scale XGBoost tuning exercise. It is a laptop-friendly XGBoost benchmark suitable for this project.


In [ ]:
# Model comparison
# IMPORTANT: Keep this cell lightweight for an 8 GB laptop.

# Calculate class imbalance ratio for XGBoost.
negative = int((y_train == 0).sum())
positive = int((y_train == 1).sum())
scale_pos_weight = negative / max(positive, 1)

print("XGBoost scale_pos_weight:", round(scale_pos_weight, 3))

models = {
    "Logistic Regression": Pipeline([
        ("prep", linear_preprocessor),
        ("model", LogisticRegression(
            max_iter=100,
            class_weight="balanced",
            solver="lbfgs",
            random_state=RANDOM_STATE
        ))
    ]),

    "XGBoost": Pipeline([
        ("prep", tree_preprocessor),
        ("model", XGBClassifier(
            n_estimators=120,
            learning_rate=0.05,
            max_depth=3,
            min_child_weight=20,
            subsample=0.80,
            colsample_bytree=0.70,
            reg_alpha=0.10,
            reg_lambda=5.0,
            gamma=0.0,
            objective="binary:logistic",
            eval_metric="auc",
            tree_method="hist",
            max_bin=64,
            device="cpu",
            n_jobs=XGB_N_JOBS,
            scale_pos_weight=scale_pos_weight,
            random_state=RANDOM_STATE
        ))
    ])
}

results = []
fitted_models = {}
probabilities = {}

for name, model in models.items():

    print("\nTraining:", name)

    model.fit(X_train, y_train)

    print("Predicting:", name)
    prob = model.predict_proba(X_test)[:, 1]
    pred = (prob >= 0.50).astype("int8")

    row = {
        "Model": name,
        "ROC_AUC": roc_auc_score(y_test, prob),
        "PR_AUC": average_precision_score(y_test, prob),
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0)
    }

    results.append(row)
    fitted_models[name] = model
    probabilities[name] = prob

    print("Finished:", name)
    print(
        f"ROC-AUC={row['ROC_AUC']:.4f} | "
        f"PR-AUC={row['PR_AUC']:.4f} | "
        f"F1={row['F1']:.4f}"
    )

    # Release temporary references between models where possible.
    gc.collect()

results_df = (
    pd.DataFrame(results)
    .sort_values("ROC_AUC", ascending=False)
    .reset_index(drop=True)
)

display(results_df)


## 12. Best-model evaluation and threshold analysis

In [ ]:
best_name = results_df.iloc[0]["Model"]
best_model = fitted_models[best_name]
best_prob = probabilities[best_name]

print("Best model:", best_name)
print(classification_report(y_test, (best_prob >= 0.50).astype(int), zero_division=0))

threshold_rows = []
for t in np.arange(0.20, 0.81, 0.10):
    pred = (best_prob >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    threshold_rows.append({
        "Threshold": round(float(t), 2),
        "Approval_Rate": round(float((pred == 0).mean()), 4),
        "Default_Precision": round(float(precision_score(y_test, pred, zero_division=0)), 4),
        "Default_Recall": round(float(recall_score(y_test, pred, zero_division=0)), 4),
        "F1": round(float(f1_score(y_test, pred, zero_division=0)), 4),
        "False_Negatives": int(fn),
        "False_Positives": int(fp)
    })

threshold_df = pd.DataFrame(threshold_rows)
display(threshold_df)

In [ ]:
# ROC and Precision-Recall curves
fpr, tpr, _ = roc_curve(y_test, best_prob)
precision, recall, _ = precision_recall_curve(y_test, best_prob)

plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label=f"ROC-AUC = {roc_auc_score(y_test, best_prob):.3f}")
plt.plot([0, 1], [0, 1], "--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC Curve — {best_name}")
plt.legend()
plt.show()

plt.figure(figsize=(6, 4))
plt.plot(recall, precision, label=f"PR-AUC = {average_precision_score(y_test, best_prob):.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(f"Precision-Recall Curve — {best_name}")
plt.legend()
plt.show()

## 13. Customer risk / eligibility segmentation

In [ ]:
def risk_segment(p):
    if p < 0.20:
        return "Eligible / Lower Risk"
    elif p < 0.40:
        return "Manual Review"
    return "High Risk"

segments = pd.DataFrame({
    "SK_ID_CURR": model_df.loc[X_test.index, "SK_ID_CURR"].values,
    "Actual_TARGET": y_test.values,
    "Predicted_Default_Probability": best_prob
})
segments["Risk_Segment"] = segments["Predicted_Default_Probability"].apply(risk_segment)

summary = (
    segments["Risk_Segment"]
    .value_counts()
    .rename_axis("Risk_Segment")
    .to_frame("Customers")
)
summary["Percentage"] = 100 * summary["Customers"] / len(segments)
display(summary)

display(segments.sort_values("Predicted_Default_Probability").head(10))
display(segments.sort_values("Predicted_Default_Probability", ascending=False).head(10))

## 14. Lightweight model interpretability

In [ ]:
# Lightweight model interpretability

if best_name == "Logistic Regression":

    fitted = best_model.named_steps["model"]
    prep = best_model.named_steps["prep"]

    transformed_names = prep.get_feature_names_out()

    coef_df = pd.DataFrame({
        "feature": transformed_names,
        "coefficient": fitted.coef_[0],
        "absolute_coefficient": np.abs(fitted.coef_[0])
    }).sort_values(
        "absolute_coefficient",
        ascending=False
    )

    display(coef_df.head(20))

    top = coef_df.head(15).sort_values("coefficient")

    plt.figure(figsize=(8, 6))
    plt.barh(top["feature"], top["coefficient"])
    plt.title("Top Logistic Regression Drivers")
    plt.xlabel("Coefficient")
    plt.tight_layout()
    plt.show()

else:
    # XGBoost provides native gain-based feature importance.
    # This is much cheaper than permutation importance on the full test set.
    fitted_xgb = best_model.named_steps["model"]

    importance_df = pd.DataFrame({
        "feature": selected_features,
        "importance_gain": fitted_xgb.feature_importances_
    }).sort_values(
        "importance_gain",
        ascending=False
    )

    display(importance_df.head(20))

    top = importance_df.head(15).sort_values("importance_gain")

    plt.figure(figsize=(8, 6))
    plt.barh(top["feature"], top["importance_gain"])
    plt.title("Top XGBoost Feature Importance")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.show()


## 15. Final Model Comparison Report

In [ ]:
final_comparison = results_df.reset_index(drop=True)
display(final_comparison)

print("Recommended production candidate:", final_comparison.iloc[0]["Model"])
print(
    "Selection basis: highest observed ROC-AUC, while also considering "
    "PR-AUC, recall, precision, explainability, threshold policy and business cost."
)

## 16. Challenges Faced and Techniques Used

### Challenge 1 — Very large relational history
The historical tables contain many rows per applicant and monthly records.

**Technique:** select only business-relevant columns, aggregate one source at a time, merge at applicant level, delete the raw table and call garbage collection.

### Challenge 2 — 8 GB RAM constraint
The original workflow could retain multiple historical tables and a large modeling matrix simultaneously.

**Technique:** sequential processing, compact `mean/min/max` aggregates, a curated 33-feature numeric model matrix, `float32` data and no large one-hot matrix.

### Challenge 3 — Class imbalance
Default is the minority class.

**Technique:** stratified train/test split, balanced Logistic Regression and XGBoost `scale_pos_weight`. ROC-AUC and PR-AUC are reported instead of relying only on accuracy.

### Challenge 4 — XGBoost memory/CPU usage
XGBoost can become resource intensive when using deep trees, many estimators, large `max_bin` values or many CPU threads.

**Technique:** `tree_method="hist"`, `max_depth=3`, `n_estimators=120`, `max_bin=64`, row/column subsampling, regularization and only two CPU threads.

### Challenge 5 — Lending decisions depend on probability thresholds
A 0.50 threshold is not necessarily the correct business policy.

**Technique:** threshold analysis showing approval rate, default precision, default recall, F1, false positives and false negatives.

### Challenge 6 — Explainability
Credit-risk decisions require understandable drivers.

**Technique:** Logistic Regression coefficients or XGBoost's native gain-based feature importance, avoiding expensive full-data permutation importance.

The project brief explicitly asks for model comparison and a report on challenges and techniques used.

## 17. Production Recommendation

For a production credit-risk system, this notebook should be treated as the modeling prototype rather than the final serving system.

Recommended architecture:

**Raw CSV/database → validation → applicant-level feature pipeline → trained model → probability score → calibration → threshold/policy engine → risk segment → monitoring**

Suggested business interpretation:

- **Eligible / Lower Risk:** lower predicted probability of default; can proceed subject to policy checks.
- **Manual Review:** intermediate risk; send for additional underwriting checks.
- **High Risk:** higher predicted probability; normally reject or require stronger controls.

### Model selection
The final candidate should be selected from `final_comparison` using ROC-AUC and PR-AUC together with recall, precision, calibration, explainability, stability and business cost.

XGBoost is included as the nonlinear challenger because it can model interactions and nonlinear patterns. Logistic Regression remains the interpretable benchmark.

Before production, validate:
- probability calibration
- fairness
- temporal stability and drift
- feature leakage
- reject-inference considerations
- business cost of false positives/false negatives
- regulatory requirements


## 18. Submission Checklist

- [x] Banking/home-loan domain analysis
- [x] Dataset and target description
- [x] Data-quality analysis
- [x] Target/class-imbalance analysis
- [x] Lightweight EDA
- [x] Historical feature engineering
- [x] Train/test split
- [x] Memory-safe preprocessing
- [x] Logistic Regression baseline
- [x] XGBoost model
- [x] Multiple-model comparison
- [x] Threshold analysis
- [x] Customer risk/eligibility segmentation
- [x] Model interpretability
- [x] Final model comparison report
- [x] Challenges and techniques report
- [x] Production recommendation
- [x] Optimized for an 8 GB laptop

**Important:** Keep the seven CSV files in the `data` folder, or keep the supplied ZIP beside the notebook, before running from Section 3 onward.

**XGBoost requirement:** If XGBoost is not installed, run `%pip install xgboost` once, restart the kernel, and rerun the notebook from the beginning.
